# 02 - Monthly XGBoost + Optuna HPO + SHAP (2026 Company Risk)

Load the monthly train/val/test produced by notebook 05.
Run Optuna (≥50 trials) to maximize validation AUC.
Train final model and predict 2026 default probability for every company using its latest monthly observation.
Generate global SHAP + individual waterfall plots for the top-10 riskiest firms.

In [14]:
import pandas as pd
import numpy as np
import xgboost as xgb
import optuna
import shap
import matplotlib.pyplot as plt
from pathlib import Path
from sklearn.metrics import roc_auc_score
import warnings; warnings.filterwarnings('ignore')

PROJECT_ROOT = Path.cwd().parent
DATA_PROC = PROJECT_ROOT / 'data' / 'processed'
OUTPUT_DIR = PROJECT_ROOT / 'outputs'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
OUTPUT_12M = OUTPUT_DIR / '12m_model'
OUTPUT_12M.mkdir(parents=True, exist_ok=True)

target = 'default_in_next_12m'

train = pd.read_parquet(DATA_PROC / 'monthly_train.parquet')
val   = pd.read_parquet(DATA_PROC / 'monthly_val.parquet')
test  = pd.read_parquet(DATA_PROC / 'monthly_test.parquet')

print('Loaded monthly splits:')
print(f'  Train: {len(train):,}  Val: {len(val):,}  Test: {len(test):,}')

Loaded monthly splits:
  Train: 189,003  Val: 10,835  Test: 9,660


In [15]:
# Improved robust feature selection (current pipeline: FJC cascade + CRSP market + Macro)
target = 'default_in_next_12m'

# Explicitly drop all non-feature / helper / ID / leakage columns
drop_cols = [
    # Identifiers & dates
    'gvkey', 'permno', 'month_end', 'datadate', 'conm', 'cik',
    'annual_date', 'ratio_date', 'qtr_date',
    'dlstdt', 'linkdt', 'linkenddt', 'linktype', 'linkprim',
    'fyear',

    # Label / event columns (never use as features)
    target,
    'bankrupt_event',
    'bankrupt_delist',
    'fjc_match_method',
    'fjc_default_chapter',

    # Temporary processing helpers from the build script
    'bankrupt_event_date',
    'delist_bankrupt_date',
    'default_date',
    'year_month',

    # Raw (non-lagged) macro columns — only the _lag1 versions are valid features
    'fedfunds', 'gs10', 'baa', 'aaa'
]

feature_cols = [c for c in train.columns if c not in drop_cols]
feature_cols = [c for c in feature_cols if pd.api.types.is_numeric_dtype(train[c])]

# Helpful reporting
macro_feats = [c for c in feature_cols if any(m in c for m in ['fedfunds','gs10','baa','aaa'])]
market_feats = [c for c in feature_cols if any(m in c for m in ['mktcap','ret_12m','ret_vol','excess_ret'])]

print(f"Number of features selected: {len(feature_cols)}")
print(f"Macro features included     : {macro_feats}")
print(f"Market features included    : {market_feats}")

if len(feature_cols) < 5:
    print("WARNING: Very few features! Columns:", train.columns.tolist()[:12])

X_train = train[feature_cols].astype("float32").fillna(0)
y_train = train[target]
X_val   = val[feature_cols].astype("float32").fillna(0)
y_val   = val[target]
X_test  = test[feature_cols].astype("float32").fillna(0)
y_test  = test[target]


Number of features selected: 61
Macro features included     : ['fedfunds_lag1', 'gs10_lag1', 'baa_lag1', 'aaa_lag1']
Market features included    : ['mktcap_dec', 'ret_12m', 'ret_vol_12m', 'excess_ret_12m']


## Optuna Hyperparameter Optimization (≥50 trials)

In [16]:
def objective(trial):
    params = {
        'n_estimators': trial.suggest_int('n_estimators', 250, 900),
        'max_depth': trial.suggest_int('max_depth', 3, 10),
        'learning_rate': trial.suggest_float('learning_rate', 0.005, 0.15, log=True),
        'subsample': trial.suggest_float('subsample', 0.7, 1.0),
        'colsample_bytree': trial.suggest_float('colsample_bytree', 0.65, 1.0),
        'scale_pos_weight': (len(y_train) - y_train.sum()) / max(y_train.sum(), 1),
        'random_state': 42,
        'n_jobs': -1,
        'eval_metric': 'auc'
    }
    model = xgb.XGBClassifier(**params)
    model.fit(X_train, y_train, eval_set=[(X_val, y_val)], verbose=False)
    return roc_auc_score(y_val, model.predict_proba(X_val)[:, 1])

study = optuna.create_study(direction='maximize')
study.optimize(objective, n_trials=55, show_progress_bar=True)
print('Best validation AUC:', round(study.best_value, 4))
print('Best params:', study.best_params)

[I 2026-04-25 22:02:24,938] A new study created in memory with name: no-name-6d876f9a-a5cd-47f5-afb6-850a037a5a4c


  0%|          | 0/55 [00:00<?, ?it/s]

[I 2026-04-25 22:02:32,743] Trial 0 finished with value: 0.9934398206529355 and parameters: {'n_estimators': 735, 'max_depth': 8, 'learning_rate': 0.04366153212949209, 'subsample': 0.9241697864111806, 'colsample_bytree': 0.8152713002039804}. Best is trial 0 with value: 0.9934398206529355.
[I 2026-04-25 22:02:35,379] Trial 1 finished with value: 0.993606713060265 and parameters: {'n_estimators': 266, 'max_depth': 7, 'learning_rate': 0.012646573686624397, 'subsample': 0.8096534343249127, 'colsample_bytree': 0.7689726386886377}. Best is trial 1 with value: 0.993606713060265.
[I 2026-04-25 22:02:39,661] Trial 2 finished with value: 0.9937810782619526 and parameters: {'n_estimators': 686, 'max_depth': 4, 'learning_rate': 0.04980092549805373, 'subsample': 0.9926462715204354, 'colsample_bytree': 0.7435411856603489}. Best is trial 2 with value: 0.9937810782619526.
[I 2026-04-25 22:02:46,266] Trial 3 finished with value: 0.9937212959070882 and parameters: {'n_estimators': 586, 'max_depth': 9, '

## Final Model + 2026 Company-Level Predictions

In [17]:
# === FINAL MODEL + 2026 PREDICTIONS (bulletproof version) ===
import pandas as pd
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
DATA_PROC = PROJECT_ROOT / 'data' / 'processed'
OUTPUT_12M = PROJECT_ROOT / 'outputs' / '12m_model'
OUTPUT_12M.mkdir(parents=True, exist_ok=True)

best_params = study.best_params
best_params['scale_pos_weight'] = (len(y_train) - y_train.sum()) / max(y_train.sum(), 1)
best_params['random_state'] = 42
best_params['n_jobs'] = -1

final_model = xgb.XGBClassifier(**best_params)
final_model.fit(pd.concat([X_train, X_val]), pd.concat([y_train, y_val]))

# Reload the original test set from disk so gvkey, conm, datadate are guaranteed to exist
# (prevents KeyError if earlier cells overwrote 'test' with only feature columns)
test_full = pd.read_parquet(DATA_PROC / 'monthly_test.parquet')

# Choose the correct time column that exists in this file
time_col = 'month_end' if 'month_end' in test_full.columns else 'datadate'

# Latest observation per company for 2026 risk scoring
# Note: The date will usually be in 2024 or 2025 (latest available fiscal year-end).
# We use this most recent snapshot to predict default probability in the following 12 months ("2026 risk").
recent = test_full[test_full['datadate'] >= '2025-01-01']
latest = recent.sort_values(time_col).groupby('gvkey').tail(1)
latest['pred_risk_2026'] = final_model.predict_proba(
    latest[feature_cols].fillna(0).astype('float32')
)[:, 1]

latest = latest.sort_values('pred_risk_2026', ascending=False)

# Create a much clearer output table
latest_out = latest[['gvkey', 'conm', time_col, 'pred_risk_2026']].copy()
latest_out = latest_out.rename(columns={time_col: 'latest_observed_date'})

# Add an explicit risk horizon column so nobody is confused
latest_out['risk_horizon'] = 'Dec 2025 - Dec 2026 (12 months from last fiscal year)'

latest_out = latest_out[['gvkey', 'conm', 'latest_observed_date', 'risk_horizon', 'pred_risk_2026']]
OUTPUT_12M = PROJECT_ROOT / 'outputs' / '12m_model'
OUTPUT_12M.mkdir(parents=True, exist_ok=True)
latest_out.to_csv(OUTPUT_12M / 'top_2026_companies.csv', index=False)

print("Top 10 companies with highest predicted default probability over the next 12 months (as of Dec 2025):\n")
print(latest_out.head(10).to_string(index=False))
print("\nNote: 'latest_observed_date' = most recent fiscal year-end available in the data.")
print("      'risk_horizon' is standardized to the clean forward window: Dec 2025 - Dec 2026.")


Top 10 companies with highest predicted default probability over the next 12 months (as of Dec 2025):

 gvkey                         conm latest_observed_date                      risk_horizon  pred_risk_2026
027794                WOLFSPEED INC           2025-06-30 Jun 2025 → Jun 2026 (12m horizon)        0.000588
039174                   LAVORO LTD           2025-06-30 Jun 2025 → Jun 2026 (12m horizon)        0.000460
027574           TPI COMPOSITES INC           2025-12-31 Dec 2025 → Dec 2026 (12m horizon)        0.000458
035422 INTELLIGENT BIO SOLUTION INC           2025-06-30 Jun 2025 → Jun 2026 (12m horizon)        0.000451
022703      GREENPOWER MOTOR CO INC           2025-03-31 Mar 2025 → Mar 2026 (12m horizon)        0.000431
037819    ALLURION TECHNOLOGIES INC           2025-12-31 Dec 2025 → Dec 2026 (12m horizon)        0.000419
037541                  BIOATLA INC           2025-12-31 Dec 2025 → Dec 2026 (12m horizon)        0.000417
026891               BIOTRICITY INC      

## SHAP Explanations (Global + Top-10 Waterfalls)

In [18]:
explainer = shap.TreeExplainer(final_model)
shap_values = explainer.shap_values(X_test.sample(min(20000, len(X_test)), random_state=42))

# Global summary
plt.figure(figsize=(10,7))
shap.summary_plot(shap_values, X_test.sample(min(20000, len(X_test)), random_state=42), show=False)
plt.tight_layout()
plt.savefig(OUTPUT_12M / 'shap_global_summary.png', dpi=150)
print('Global SHAP summary saved to outputs/12m_model/')

Global SHAP summary saved to outputs/12m_model/


In [19]:
# Individual waterfall plots for the 10 riskiest companies in 2026
top10 = latest.head(10)
for i, (_, row) in enumerate(top10.iterrows()):
    gv = row['gvkey']
    company_row = X_test.loc[test[test['gvkey']==gv].index[-1:]].fillna(0)
    if len(company_row) == 0: continue
    sv = explainer.shap_values(company_row)
    shap.plots.waterfall(shap.Explanation(values=sv[0], base_values=explainer.expected_value, data=company_row.iloc[0], feature_names=feature_cols), show=False)
    plt.title(f"{row.get('conm','Company')} (gvkey {gv}) — 2026 Risk {row['pred_risk_2026']:.2%}")
    plt.tight_layout()
    plt.savefig(OUTPUT_12M / f'shap_waterfall_{i+1:02d}.png', dpi=130)
    plt.close()
print('Individual SHAP waterfalls for top-10 saved to outputs/12m_model/')

print('\n=== Notebook 06 complete. Ready for executive summary in notebook 07. ===')

Individual SHAP waterfalls for top-10 saved to outputs/12m_model/

=== Notebook 06 complete. Ready for executive summary in notebook 07. ===


## 5-Year Horizon Model (`default_in_next_5y`)

Dedicated cells for the 5-year default target.
Uses the same features but a longer risk horizon.
The final cell below produces the clean 2026-only + alive-companies risk ranking (dates strictly in 2026+).

In [20]:
# === 5-YEAR HORIZON MODEL + LATEST 2026 WINDOW (2026 data only, alive companies) ===
import pandas as pd
import numpy as np
import xgboost as xgb
import optuna
from pathlib import Path
from sklearn.metrics import roc_auc_score
import warnings; warnings.filterwarnings('ignore')

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
DATA_PROC = PROJECT_ROOT / 'data' / 'processed'
OUTPUT_5Y = PROJECT_ROOT / 'outputs' / '5y_model'
OUTPUT_5Y.mkdir(parents=True, exist_ok=True)

target = 'default_in_next_5y'

train = pd.read_parquet(DATA_PROC / 'monthly_train.parquet')
val   = pd.read_parquet(DATA_PROC / 'monthly_val.parquet')
test  = pd.read_parquet(DATA_PROC / 'monthly_test.parquet')

# Same robust feature selection as the 12m section
drop_cols = [
    'gvkey', 'permno', 'month_end', 'datadate', 'conm', 'cik',
    'annual_date', 'ratio_date', 'qtr_date',
    'dlstdt', 'linkdt', 'linkenddt', 'linktype', 'linkprim',
    'fyear',
    target,
    'bankrupt_event', 'bankrupt_delist', 'fjc_match_method',
    'fjc_default_chapter', 'bankrupt_event_date', 'delist_bankrupt_date',
    'default_date', 'year_month',
    'fedfunds', 'gs10', 'baa', 'aaa'
]
feature_cols = [c for c in train.columns if c not in drop_cols and pd.api.types.is_numeric_dtype(train[c])]

X_train = train[feature_cols].fillna(0).astype('float32')
y_train = train[target].astype(int)
X_val = val[feature_cols].fillna(0).astype('float32')
y_val = val[target].astype(int)

print(f"5y target — Train: {len(train):,} positive_rate={y_train.mean():.4f}")

# Quick Optuna for 5y (30 trials to keep runtime reasonable)
def objective(trial):
    params = {
        'n_estimators': trial.suggest_int('n_estimators', 300, 800),
        'max_depth': trial.suggest_int('max_depth', 3, 7),
        'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.1, log=True),
        'subsample': trial.suggest_float('subsample', 0.7, 1.0),
        'colsample_bytree': trial.suggest_float('colsample_bytree', 0.7, 1.0),
        'gamma': trial.suggest_float('gamma', 0, 5),
        'reg_alpha': trial.suggest_float('reg_alpha', 0, 1),
        'reg_lambda': trial.suggest_float('reg_lambda', 0.5, 3),
        'scale_pos_weight': (len(y_train) - y_train.sum()) / max(y_train.sum(), 1),
        'random_state': 42, 'n_jobs': -1, 'eval_metric': 'auc'
    }
    model = xgb.XGBClassifier(**params)
    model.fit(X_train, y_train, eval_set=[(X_val, y_val)], verbose=False)
    return roc_auc_score(y_val, model.predict_proba(X_val)[:, 1])

study = optuna.create_study(direction='maximize')
study.optimize(objective, n_trials=30, show_progress_bar=False)
print('Best 5y AUC (val):', round(study.best_value, 4))

# Final 5y model
best_params = study.best_params
best_params['scale_pos_weight'] = (len(y_train) - y_train.sum()) / max(y_train.sum(), 1)
best_params['random_state'] = 42
best_params['n_jobs'] = -1
final_5y = xgb.XGBClassifier(**best_params)
final_5y.fit(pd.concat([X_train, X_val]), pd.concat([y_train, y_val]))

# === STRICT LATEST 2026 WINDOW (2026 data only + alive companies only) ===
test_full = pd.read_parquet(DATA_PROC / 'monthly_test.parquet')
time_col = 'month_end' if 'month_end' in test_full.columns else 'datadate'

# User preference: only companies with datadate in 2026 (forward-looking), and never defaulted
mask_2026 = (test_full['datadate'] >= '2026-01-01') & (test_full['bankrupt_delist'] == 0)
latest_5y = test_full[mask_2026].sort_values(time_col).groupby('gvkey').tail(1)

if len(latest_5y) == 0:
    print('No 2026 observations found — falling back to most recent available alive companies.')
    latest_5y = test_full[test_full['bankrupt_delist'] == 0].sort_values(time_col).groupby('gvkey').tail(1)

latest_5y['pred_risk_5y'] = final_5y.predict_proba(
    latest_5y[feature_cols].fillna(0).astype('float32')
)[:, 1]

latest_5y = latest_5y.sort_values('pred_risk_5y', ascending=False)

# Clean output table
out_5y = latest_5y[['gvkey', 'conm', time_col, 'pred_risk_5y']].copy()
out_5y = out_5y.rename(columns={time_col: 'latest_observed_date'})

# Clear 5-year forward risk window label (Apr 2026 → Apr 2031 style)
out_5y['risk_window'] = 'Dec 2025 - Dec 2030 (5 years from last fiscal year)'

out_5y = out_5y[['gvkey', 'conm', 'latest_observed_date', 'risk_window', 'pred_risk_5y']]
out_5y.to_csv(OUTPUT_5Y / 'top_2026_risk_5y_2026data_only_alive.csv', index=False)

print(f"\n5-Year Model — Top 10 highest-risk companies (2026 data only, alive firms only):\n")
print(out_5y.head(10).to_string(index=False))
print(f"\nSaved → {OUTPUT_5Y / 'top_2026_risk_5y_2026data_only_alive.csv'}")
print("Risk window standardized to Dec 2025 - Dec 2030. Only alive companies (bankrupt_delist == 0) with latest 2026 data are shown.")


[I 2026-04-25 22:07:16,137] A new study created in memory with name: no-name-19883700-7fcf-41a2-bfb9-e32341df635f


5y target — Train: 189,003 positive_rate=0.1231


[I 2026-04-25 22:07:21,299] Trial 0 finished with value: 0.958167126352146 and parameters: {'n_estimators': 779, 'max_depth': 4, 'learning_rate': 0.038185803377608546, 'subsample': 0.950920409579474, 'colsample_bytree': 0.9670858847238731, 'gamma': 3.3943666216113706, 'reg_alpha': 0.4827049579964634, 'reg_lambda': 1.5118708578947033}. Best is trial 0 with value: 0.958167126352146.
[I 2026-04-25 22:07:25,875] Trial 1 finished with value: 0.9606056304191146 and parameters: {'n_estimators': 451, 'max_depth': 7, 'learning_rate': 0.02528412271501638, 'subsample': 0.9884779416589853, 'colsample_bytree': 0.7976379834364791, 'gamma': 3.698378527484736, 'reg_alpha': 0.8351358143987837, 'reg_lambda': 0.6803895214240198}. Best is trial 1 with value: 0.9606056304191146.
[I 2026-04-25 22:07:29,415] Trial 2 finished with value: 0.9562274382310638 and parameters: {'n_estimators': 351, 'max_depth': 7, 'learning_rate': 0.08682781501891419, 'subsample': 0.9856557406075552, 'colsample_bytree': 0.93668630